# ⚙️ Notebook 5: Feature Engineering
**Project:** TalentSight — Employee Attrition Intelligence Platform  
**Author:** Saheri  
**Date:** June 2026  

## Objective
Transform four cleaned data sources into a single, model-ready feature matrix. 
This notebook covers:
1. **Source cleaning** — BLS data parsing, synthetic payroll messsiness handling
2. **Multi-source merging** — joining payroll with BLS macro context before aggregation
3. **Payroll aggregation** — collapsing 24 monthly rows into 4 employee-level features
4. **Domain feature engineering** — 5 novel features built from HR domain knowledge
5. **Preprocessing pipeline** — ColumnTransformer encoding all feature types correctly
6. **Feature selection** — multicollinearity removal based on correlation analysis

**Output:** `data/processed/employee_master_dataset.csv` — 1,470 rows × 29 features, 
ready for modelling in `06_Modelling.ipynb`.

## 0. Setup & Libraries
All preprocessing tools are imported upfront. `PowerTransformer` is used instead 
of `StandardScaler` for numerical features — Yeo-Johnson handles both positive and 
negative values and reduces skew more effectively than simple standardisation for 
right-skewed distributions like `MonthlyIncome` and `YearsSinceLastPromotion`.

In [26]:
#Import Necessary Libraries
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer,OrdinalEncoder
from sklearn.compose import ColumnTransformer
import joblib

## 1. Load All Data Sources
Four cleaned datasets are loaded:
- **IBM HR dataset** — 1,470 employee records, primary source
- **Synthetic payroll** — 35,280 rows (24 months × 1,470 employees), messy version
- **BLS labor data** — 84 rows of national macro indicators (2023–2025)

Note: BLS and ECI data are cleaned in this notebook before merging, since their 
cleaning logic belongs with their usage context rather than in a separate notebook.

In [27]:
# Load the datasets
BASE_DIR = os.getcwd()

ibm = pd.read_csv(os.path.join(BASE_DIR,'..','data', 'processed', 'cleaned_employee_attrition.csv'))

payroll = pd.read_csv(os.path.join(BASE_DIR,'..','data', 'processed', 'synthetic_Payroll_24_Months.csv'))

bls = pd.read_csv(os.path.join(BASE_DIR,'..','data', 'raw', 'bls_labor_data.csv'))

## 2. Clean External Data Sources

### 2.1 BLS Data Cleaning
The raw BLS dataset contains three series with different time granularities:
- `JTS000000000000000QUR` — JOLTS Quits Rate (monthly)
- `LNS14000000` — Unemployment Rate (monthly)  
- `CIU1010000000000A` — Employment Cost Index (quarterly)

These are split into two separate dataframes — monthly and quarterly — 
since forcing different time granularities into one table would require 
fabricating precision that doesn't exist in the source data.

**Cleaning decisions:**
- `Period` column (`M01`–`M12`) mapped to integer month numbers for joining with payroll
- Suppressed BLS value (`-`) in October 2025 unemployment replaced with `NaN` — 
  this is a known BLS confidentiality suppression, not a data error
- Quarterly ECI periods (`Q01`–`Q04`) mapped to integer quarter numbers

In [28]:
#Split the BLS data into three separate dataframes based on the 'SeriesID' column
bls_quits_rates = bls[bls['SeriesID'] == 'JTS000000000000000QUR'].copy()  # Quite Rates
bls_unemployment_rates = bls[bls['SeriesID'] == 'LNS14000000'].copy()  # Unemployment Rates
bls_employment_cost = bls[bls['SeriesID'] == 'CIU1010000000000A'].copy()  # employment cost index

In [29]:
# Merge the bls_quits_rates and bls_unemployment_rates dataframes on the 'Year' and 'Period' column
bls_merged_monthly = pd.merge(bls_quits_rates, bls_unemployment_rates, on=['Year', 'Period'], suffixes=('_quits_rates', '_unemployment_rates'))

In [30]:
# Change the name of the 'Value' column in bls_employment_cost to 'Employment_Cost_Index'
bls_employment_cost.rename(columns={'Value': 'Employment_Cost_Index'}, inplace=True)

#Change the name of the 'Value' column in bls_merged_monthly to 'Quits_Rates' and 'Unemployment_Rates'
bls_merged_monthly.rename(columns={'Value_quits_rates': 'Quits_Rates', 'Value_unemployment_rates': 'Unemployment_Rates', 'PeriodName_quits_rates': 'Month'}, inplace=True) 

In [31]:
# Drop unnecessary columns from bls_merged_monthly
bls_merged_monthly.drop(columns=['SeriesID_quits_rates', 'SeriesID_unemployment_rates', 'Period', 'PeriodName_unemployment_rates'], inplace=True)

In [32]:
# Map the 'Month' column in bls_merged_monthly to numerical values
month_mapping = {
    'January': 1,
    'February': 2,
    'March': 3,
    'April': 4,
    'May': 5,
    'June': 6,
    'July': 7,
    'August': 8,
    'September': 9,
    'October': 10,
    'November': 11,
    'December': 12
}
bls_merged_monthly['Month'] = bls_merged_monthly['Month'].map(month_mapping)

In [ ]:
# Display the information of the merged BLS dataframe
bls_merged_monthly.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Year                36 non-null     int64
 1   Quits_Rates         36 non-null     str  
 2   Month               36 non-null     int64
 3   Unemployment_Rates  36 non-null     str  
dtypes: int64(2), str(2)
memory usage: 1.5 KB


In [34]:
# Change the datatype of Quits_Rates to float
bls_merged_monthly['Quits_Rates'] = bls_merged_monthly['Quits_Rates'].astype(float)
# Replace the - in the Unemployment_Rates column with NaN and convert the datatype to float
bls_merged_monthly['Unemployment_Rates'] = bls_merged_monthly['Unemployment_Rates'].replace('-', np.nan).astype(float)

In [35]:
# Drop unnecessary columns from bls_employment_cost
bls_employment_cost.drop(columns=['SeriesID', 'PeriodName'], inplace=True)

In [36]:
# Change the name of 'Period' column in bls_employment_cost to 'Quarter'
bls_employment_cost.rename(columns={'Period': 'Quarter'}, inplace=True)
#Map the 'Quarter' column in bls_employment_cost to numerical values
quarter_mapping = {
    'Q01': 1,
    'Q02': 2,
    'Q03': 3,
    'Q04': 4
}
bls_employment_cost['Quarter'] = bls_employment_cost['Quarter'].map(quarter_mapping)

In [37]:
# Change the datatype of Employment_Cost_Index to float
bls_employment_cost['Employment_Cost_Index'] = bls_employment_cost['Employment_Cost_Index'].astype(float)

In [38]:
# Save the cleaned datasets to the processed folder
bls_merged_monthly.to_csv(os.path.join(BASE_DIR, '..', 'data', 'processed', 'bls_monthly_macro.csv'), index=False)
bls_employment_cost.to_csv(os.path.join(BASE_DIR, '..', 'data', 'processed', 'bls_quarterly_eci.csv'), index=False)

### Finding: BLS Macro Data — Modelling vs Dashboard Decision
After cleaning, both monthly BLS series (`avg_quits_rate`, `avg_unemployment_rate`) 
were tested as model features. Since all 1,470 employees share the same 24-month 
payroll window (2024–2025), averaging BLS values per employee produces near-constant 
values (std < 0.01) with zero discriminative power for the model.

**Decision:** BLS macro data is excluded from the model feature set and redirected 
to the Power BI dashboard, where it serves as an external benchmarking layer — 
comparing internal attrition trends against national labour market conditions.

This is consistent with how macro data is used in professional HR analytics: 
as context for business narrative, not as individual-level predictors.

In [39]:
# Checking the Synthetic Payroll Data
print("Duplicate rows:", payroll.duplicated().sum())
print("Missing OvertimePay:", payroll['OvertimePay'].isna().sum())
print("Bonus describe:\n", payroll['Bonus'].describe())

Duplicate rows: 94
Missing OvertimePay: 1769
Bonus describe:
 count    35380.000000
mean       163.317633
std        399.089458
min          0.000000
25%          0.000000
50%          0.000000
75%        155.300000
max      12190.100000
Name: Bonus, dtype: float64


### 2.2 Synthetic Payroll Cleaning
The synthetic payroll dataset was deliberately generated with three types of 
messiness to simulate a realistic HRIS export:

| Issue | Count | Business Interpretation |
|---|---|---|
| Duplicate rows | 94 | Data entry errors in payroll system |
| Missing `OvertimePay` | 1,769 (~5%) | Unrecorded overtime — administrative oversight |
| Outlier bonuses | 5 rows | 10× multiplication error simulating digit entry mistake |

**Cleaning decisions:**
- Duplicates → dropped, keeping first occurrence
- Missing `OvertimePay` → **left as NaN intentionally** — `.mean()` in aggregation 
  skips NaN by default, giving an honest average from known months only. 
  Missingness itself becomes a feature (`months_missing_overtime` was considered 
  but excluded after SHAP analysis showed near-zero importance)
- Outlier bonuses → **capped at `BaseSalary × 0.22`** using business logic 
  (documented maximum bonus rate), not statistical IQR — IQR would incorrectly 
  penalise legitimate high-salary bonuses

In [40]:
# Drop duplicate rows from payroll
payroll.drop_duplicates(inplace=True)
print("Duplicate rows after dropping:", payroll.duplicated().sum())

Duplicate rows after dropping: 0


In [41]:
# Check bonus distribution among bonus-paying months only
bonus_paid = payroll[payroll['Bonus'] > 0]
print(bonus_paid['Bonus'].describe())

count     9740.000000
mean       592.096503
std        569.304379
min         52.390000
25%        243.430000
50%        411.255000
75%        728.315000
max      12190.100000
Name: Bonus, dtype: float64


In [42]:
# Identify outliers in the Bonus column using the IQR method
Q1 = bonus_paid['Bonus'].quantile(0.25)
Q3 = bonus_paid['Bonus'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

outliers = payroll[payroll['Bonus'] > upper_bound]
print(f"Number of outlier bonuses detected: {len(outliers)}")
print(f"Upper bound threshold: {upper_bound:.2f}")

Number of outlier bonuses detected: 736
Upper bound threshold: 1455.64


In [43]:
# Check the no of rows that exceeds bonus = current salary*0.22 
bonus_threshold = payroll['BaseSalary'] * 0.22
exceeding_bonuses = payroll[payroll['Bonus'] > bonus_threshold]
print(f"Number of rows exceeding bonus threshold: {len(exceeding_bonuses)}")

Number of rows exceeding bonus threshold: 5


In [44]:
# Cap the bonus to 22% of the base salary for outliers
exceeding_mask = payroll['Bonus'] > (payroll['BaseSalary'] * 0.22)
payroll.loc[exceeding_mask, 'Bonus'] = payroll.loc[exceeding_mask, 'BaseSalary'] * 0.22
print(f"Capped {exceeding_mask.sum()} outlier bonus rows to the 22% ceiling")

Capped 5 outlier bonus rows to the 22% ceiling


In [45]:
# Save the cleaned payroll dataset to the processed folder
payroll.to_csv(os.path.join(BASE_DIR, '..', 'data', 'processed', 'synthetic_Payroll_24_Months_cleaned.csv'), index=False)

## 3. Load Cleaned Sources and Merge

### 3.1 Join Strategy
The merge sequence follows a deliberate order to enrich payroll data with 
macro context **before** aggregating to employee level:

In [46]:
# Import the cleaned datasets for feature engineering
BASE_DIR = os.getcwd()
IBM_cleaned = pd.read_csv(os.path.join(BASE_DIR,'..','data', 'processed', 'cleaned_employee_attrition.csv'))
Payroll_cleaned = pd.read_csv(os.path.join(BASE_DIR,'..','data', 'processed', 'synthetic_Payroll_24_Months_cleaned.csv'))
bls_monthly = pd.read_csv(os.path.join(BASE_DIR,'..','data', 'processed', 'bls_monthly_macro.csv'))
bls_quarterly_eci = pd.read_csv(os.path.join(BASE_DIR,'..','data', 'processed', 'bls_quarterly_eci.csv'))

In [47]:
# Print the shapes of the cleaned datasets
print("IBM shape:", IBM_cleaned.shape)
print("Payroll shape:", Payroll_cleaned.shape)
print("BLS Monthly shape:", bls_monthly.shape)
print("BLS ECI shape:", bls_quarterly_eci.shape)

IBM shape: (1470, 32)
Payroll shape: (35286, 8)
BLS Monthly shape: (36, 4)
BLS ECI shape: (12, 3)


In [48]:
# Merge the bls_monthly and payroll dataset on 'Year' and 'Month' columns
bls_payroll_merged = pd.merge(bls_monthly, Payroll_cleaned, on=['Year', 'Month'])

Joining IBM first (before aggregation) was considered and rejected — it would 
carry 32 IBM columns through the aggregation step unnecessarily, increasing 
memory and complexity with no benefit to the final output.

In [49]:
# Verify the merged dataset
print("Merged shape:", bls_payroll_merged.shape)
print("Columns:", bls_payroll_merged.columns.tolist())
print(bls_payroll_merged.head(3))

Merged shape: (35286, 10)
Columns: ['Year', 'Quits_Rates', 'Month', 'Unemployment_Rates', 'EmployeeNumber', 'EmployeeName', 'BaseSalary', 'OvertimePay', 'Bonus', 'TotalCompensation']
   Year  Quits_Rates  Month  Unemployment_Rates  EmployeeNumber  \
0  2025          2.0     12                 4.4               1   
1  2025          2.0     12                 4.4               2   
2  2025          2.0     12                 4.4               4   

      EmployeeName  BaseSalary  OvertimePay    Bonus  TotalCompensation  
0     Allison Hill     6379.39       790.85     0.00            7170.24  
1      Noah Rhodes     5754.66         0.00  1148.22            6902.87  
2  Angie Henderson     2381.47       243.28   126.64            2751.39  


### Confirmed: Merge Successful
35,286 rows preserved after joining payroll with BLS monthly data. 
Each payroll row now carries the national quits rate and unemployment rate 
for that specific month — enabling time-aware aggregation in the next step.

## 4. Payroll Feature Aggregation
Each employee's 24 monthly payroll rows are collapsed into 4 scalar features 
that capture dimensions IBM's single-point-in-time snapshot cannot:

| Feature | Formula | What it adds beyond IBM |
|---|---|---|
| `salary_growth_rate` | (last - first BaseSalary) / first | IBM has snapshot income; this captures trajectory |
| `bonus_month_ratio` | months with Bonus > 0 / 24 | IBM has no bonus frequency data |
| `overtime_month_ratio` | months with OvertimePay > 0 / 24 | IBM's binary OverTime Yes/No loses intensity |
| `avg_bonus_pct` | total bonus / total base salary × 100 | Compensation quality relative to base pay |

**Note on sort order:** Data is sorted ascending by `[EmployeeNumber, Year, Month]` 
before aggregation to ensure `iloc[0]` captures January 2024 (earliest) and 
`iloc[-1]` captures the most recent month for `salary_growth_rate` calculation.

In [50]:
# Creating the Payroll Features
payroll_features = (
    bls_payroll_merged
    .sort_values(
        by=['EmployeeNumber', 'Year', 'Month']
    )
    .groupby('EmployeeNumber')
    .agg(
        salary_growth_rate=(
            'BaseSalary',
            lambda x: (x.iloc[-1] - x.iloc[0]) / x.iloc[0]
        ),
        total_bonus=(
            'Bonus',
            lambda x: x.sum()
        ),
        total_base_salary=(
            'BaseSalary',
            lambda x: x.sum()
        ), 
        compensation_volatility=(
            'TotalCompensation',
            'std'
        ),
       bonus_month_ratio=(
            'Bonus',
            
            lambda x: (x > 0).mean()
        ),
        total_compensation_sum=(
            'TotalCompensation',
            'sum'
        ),
        avg_total_compensation=(
            'TotalCompensation',
            'mean'
        ),
        overtime_month_ratio=(
            'OvertimePay',
            lambda x: (x.fillna(0) > 0).mean()
        ),
        total_overtime=(
            'OvertimePay',
             lambda x: x.sum()
        ),
        max_bonus=(
            'Bonus',
            'max'
        )
        
    )
    .reset_index()
)

In [51]:
# Average Bonus Percetage
payroll_features['avg_bonus_pct']= (
     payroll_features['total_bonus']
     /
     payroll_features['total_base_salary'])*100


# Drop helper column
payroll_features.drop(
    columns=['total_compensation_sum','total_bonus','total_base_salary','compensation_volatility','avg_total_compensation','total_overtime','max_bonus'],
    inplace=True
)

In [52]:
#Checking the Payroll Features
print(payroll_features.shape) 
print(payroll_features.head())

(1470, 5)
   EmployeeNumber  salary_growth_rate  bonus_month_ratio  \
0               1            0.064474           0.291667   
1               2            0.121766           0.333333   
2               4            0.139459           0.250000   
3               5            0.179495           0.250000   
4               7            0.081127           0.291667   

   overtime_month_ratio  avg_bonus_pct  
0              0.833333       2.456979  
1              0.083333       6.714265  
2              0.666667       1.358956  
3              0.625000       1.781083  
4              0.125000       1.602236  


In [53]:
# Descriptive Statistics
payroll_features.describe().T

,count,mean,std,min,25%,50%,75%,max
EmployeeNumber,1470.0,1024.865306,602.024335,1.000000,491.250000,1020.500000,1555.750000,2068.000000
salary_growth_rate,1470.0,0.115895,0.037738,0.050076,0.083528,0.116290,0.148577,0.179955
bonus_month_ratio,1470.0,0.276036,0.050471,0.083333,0.250000,0.291667,0.333333,0.333333
overtime_month_ratio,1470.0,0.270909,0.291466,0.000000,0.083333,0.125000,0.625000,0.958333
avg_bonus_pct,1470.0,2.400081,1.450508,0.397467,1.475711,1.946985,2.636440,7.418663


In [54]:
# Merging Payroll Features with the IBM Dataset
employee_master = IBM_cleaned.merge(
    payroll_features,
    on='EmployeeNumber',
    how='left'
)

In [55]:
# Checking The Merged Dataframe
print(employee_master.shape)
print(employee_master.isna().sum().sort_values(ascending=False).head())

(1470, 36)
Age               0
Attrition         0
BusinessTravel    0
DailyRate         0
Department        0
dtype: int64


## 5. Domain-Driven Feature Engineering
Five features are engineered from IBM columns using HR domain knowledge. 
These capture **ratios and interactions** that XGBoost cannot derive by itself 
from individual columns — each represents a business concept not expressible 
as a single raw feature.

| Feature | Formula | Business rationale |
|---|---|---|
| `career_stagnation_index` | YearsAtCompany / (NumCompaniesWorked + 1) | Loyalty relative to career mobility history |
| `promotion_velocity` | YearsAtCompany / (YearsSinceLastPromotion + 1) | How recently was growth rewarded? |
| `role_stagnation_ratio` | YearsInCurrentRole / (YearsAtCompany + 1) | Proportion of tenure spent in same role |
| `manager_stability_ratio` | YearsWithCurrManager / (YearsAtCompany + 1) | Relationship continuity as retention factor |
| `training_intensity` | TrainingTimesLastYear / (YearsAtCompany + 1) | Investment in employee development relative to tenure |

The `+1` in denominators prevents division by zero for employees with 0 years 
in a given dimension (e.g. new joiners).

In [56]:
# Career Stagnation Index
employee_master['career_stagnation_index'] = (
    employee_master['YearsAtCompany']
    /
    (employee_master['NumCompaniesWorked'] + 1)
)
# Promotion Velocity
employee_master['promotion_velocity'] = (
    employee_master['YearsAtCompany']
    /
    (employee_master['YearsSinceLastPromotion'] + 1)
)
# Role Stagnation Ratio
employee_master['role_stagnation_ratio'] = (
    employee_master['YearsInCurrentRole']
    /
    (employee_master['YearsAtCompany'] + 1)
)
# Manager Stability Ratio
employee_master['manager_stability_ratio'] = (
    employee_master['YearsWithCurrManager']
    /
    (employee_master['YearsAtCompany'] + 1)
)
# Training Intensity
employee_master['training_intensity'] = (
    employee_master['TrainingTimesLastYear']
    /
    (employee_master['YearsAtCompany'] + 1)
)

In [57]:
# Checking the newly created features
employee_master[
    [
        'career_stagnation_index',
        'promotion_velocity',
        'role_stagnation_ratio',
        'manager_stability_ratio',
        'training_intensity'
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
career_stagnation_index,1470.0,3.158164,4.026949,0.0,0.600000,1.75,4.500000,37.000000
promotion_velocity,1470.0,3.116007,3.116739,0.0,1.000000,2.00,4.000000,33.000000
role_stagnation_ratio,1470.0,0.480701,0.274128,0.0,0.333333,0.50,0.666667,0.882353
manager_stability_ratio,1470.0,0.465510,0.276763,0.0,0.285714,0.50,0.666667,0.894737
training_intensity,1470.0,0.623948,0.714059,0.0,0.222222,0.40,0.750000,6.000000


## 6. Feature Selection — Dropping Irrelevant and Redundant Columns

### 6.1 Dropped for ethical or statistical reasons

| Column | Reason |
|---|---|
| `Gender` | Ethical — using gender as attrition predictor risks discriminatory outputs |
| `PerformanceRating` | Near-zero variance — only values 3 and 4 exist in dataset |
| `OverTime` | Redundant — `overtime_month_ratio` captures overtime signal more precisely |
| `EducationField` | Weak signal — majority of encoded categories showed zero importance |

### 6.2 Dropped for multicollinearity (correlation > 0.7)

Correlation analysis identified 16 highly correlated feature pairs. Retaining 
all correlated features splits model importance across redundant signals, 
reducing interpretability and increasing overfitting risk.

| Dropped | Retained | Correlation | Reason |
|---|---|---|---|
| `YearsInCurrentRole` | `role_stagnation_ratio` | 0.759 | Engineered feature subsumes raw |
| `YearsWithCurrManager` | `manager_stability_ratio` | 0.769 | Engineered feature subsumes raw |
| `TotalWorkingYears` | `MonthlyIncome` | 0.773 | Income more directly actionable |

### 6.3 Dropped for ID/noise reasons

`EmployeeNumber`, `EmployeeName`, `DailyRate`, `HourlyRate`, `MonthlyRate` — 
ID columns or rate fields with inconsistent meaning confirmed by data audit.

In [58]:
# Drop unnecessary columns from employee_master
employee_master.drop(columns=['EmployeeNumber','DailyRate','HourlyRate','MonthlyRate','Gender','PerformanceRating','OverTime','EducationField','YearsWithCurrManager', 'YearsInCurrentRole',
    'TotalWorkingYears'], inplace=True)

In [59]:
#Feature and Target Separation
X = employee_master.drop('Attrition', axis=1)  
y= employee_master['Attrition'].map({'Yes': 1, 'No': 0})  # Encoding target variable

## 7. Preprocessing Pipeline (ColumnTransformer)

Features require different encoding strategies based on their measurement type:

| Transformer | Columns | Rationale |
|---|---|---|
| `OrdinalEncoder` with defined order | `BusinessTravel` | Non-Travel < Travel_Rarely < Travel_Frequently — meaningful order |
| `OrdinalEncoder` default | 7 satisfaction/level scales | Integer ordinal scales 1–4 or 1–5, order preserved |
| `OneHotEncoder(drop='first')` | `Department`, `JobRole`, `MaritalStatus` | Nominal categories, no inherent order, dummy trap avoided |
| `PowerTransformer(yeo-johnson)` | All continuous numerical features | Reduces right skew in income/tenure distributions |

**Important:** The preprocessor is fitted on training data only in `06_Modelling.ipynb` 
to prevent data leakage. Here we only define and save the unfitted transformer object.

In [60]:
# Defining Feature Types
binary_cols = [] 

onehot_cols = ['Department', 'JobRole', 'MaritalStatus']

ordinal_cols_business_travel = ['BusinessTravel']

ordinal_cols_scale = ['JobLevel', 'EnvironmentSatisfaction', 'JobSatisfaction',
                      'WorkLifeBalance', 'JobInvolvement', 'RelationshipSatisfaction',
                      'StockOptionLevel']

numerical_features = [col for col in X.columns 
                      if col not in binary_cols + onehot_cols + 
                      ordinal_cols_business_travel + ordinal_cols_scale]

In [61]:
# Applying ColumnTransformer for encoding categorical features and scaling numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), binary_cols),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), onehot_cols),
        ('ordinal_travel', OrdinalEncoder(
            categories=[['Non-Travel', 'Travel_Rarely', 'Travel_Frequently']]
        ), ordinal_cols_business_travel),
        ('ordinal_scale', OrdinalEncoder(), ordinal_cols_scale),
        ('num', PowerTransformer(method='yeo-johnson'), numerical_features),
    ],
    remainder='drop'
)

In [62]:
# Checking the columns being assigned and dropped
assigned_cols = (binary_cols + onehot_cols + 
                 ordinal_cols_business_travel + 
                 ordinal_cols_scale + 
                 numerical_features)

dropped = [col for col in X.columns if col not in assigned_cols]
print(f"Columns being silently dropped: {dropped}")
print(f"Total assigned: {len(assigned_cols)} of {X.shape[1]}")

Columns being silently dropped: []
Total assigned: 29 of 29


### Confirmed: All 29 Features Assigned
Zero columns falling through to `remainder='drop'`. Every feature is explicitly 
assigned to a transformer with documented rationale.

**Final feature set: 29 features across 3 source types**
- IBM raw features: 19
- Payroll aggregated features: 4  
- Domain engineered features: 5
- BLS features: 0 (redirected to Power BI — see Section 2 finding)

In [63]:
# Save the final employee master dataset to the processed folder
employee_master.to_csv(os.path.join(BASE_DIR, '..', 'data', 'processed', 'employee_master_dataset.csv'), index=False)

In [64]:
# Save the preprocessor to a file for future use
joblib.dump(preprocessor, os.path.join(BASE_DIR, '..', 'models', 'preprocessor.pkl'))

['c:\\Projects\\TalentSight\\talentsight\\notebooks\\..\\models\\preprocessor.pkl']

## 8. Outputs Saved

| File | Location | Description |
|---|---|---|
| `employee_master_dataset.csv` | `data/processed/` | Final feature matrix — 1,470 × 30 (29 features + Attrition) |
| `preprocessor.pkl` | `models/` | Unfitted ColumnTransformer — fitted on X_train in modelling notebook |
| `bls_monthly_macro.csv` | `data/processed/` | Cleaned BLS monthly series — for Power BI dashboard |
| `bls_quarterly_eci.csv` | `data/processed/` | Cleaned ECI quarterly series — for Power BI dashboard |
| `synthetic_Payroll_24_Months_cleaned.csv` | `data/processed/` | Cleaned payroll data |

---

**Next:** `06_Modelling.ipynb` — Logistic Regression baseline, XGBoost tuning, 
SHAP analysis, business cost threshold optimisation.